# 05. Publication

Three outputs:

1. **The files the web application reads**, as JSON, into `web/data/`
2. **The municipal boundaries**, from the IDEM shapefile into GeoJSON
3. **The whole database as RDF**, into `rdf/madrid_crime.ttl`

## Settings

In [ ]:
import os
import getpass

DB = {
    "host": "localhost",
    "port": 5432,
    "dbname": "madrid_crime",
    "user": "postgres",
    "password": os.environ.get("PGPASSWORD") or getpass.getpass("PostgreSQL password: "),
}

WEB_DIR = "../web/data"
SHAPEFILE = "../data/raw/boundaries/IDEM_CM_UNID_ADMINPolygon.shp"
GEOJSON = "../web/data/madrid_municipalities.geojson"
RDF_DIR = "../rdf"

os.makedirs(WEB_DIR, exist_ok=True)

In [ ]:
import sys

sys.path.append("..")

import json
from pathlib import Path
import warnings
import pandas as pd
import psycopg2
import geopandas as gpd
import shapely
from src.utils import q, save_table, year_range

SHAPEFILE = Path("../data/raw/boundaries/IDEM_CM_UNID_ADMINPolygon.shp")
OUTPUT_JSON = Path("../web/data/madrid_municipalities.geojson")

warnings.filterwarnings("ignore", message=".*SQLAlchemy.*")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

conn = psycopg2.connect(**DB)
conn.autocommit = True

print("connected to", DB["dbname"])

## Export for the web page

The web page is a set of static files with no database behind it, so
everything it needs has to be written out here.

In [ ]:
web = Path(WEB_DIR)
web.mkdir(parents=True, exist_ok=True)


save_table(q("""
    SELECT ine_code, year, crime_code, denominator,
           offences, rate, rate_per_1000, rate_per_100000
    FROM   ind_crime_level
""", conn), web, "crime_level")

save_table(q("""
    SELECT year, crime_code, level, parent_code, recorded, cleared, unsolved,
           clearance_rate, clearance_above_one, unsolved_share_in_level
    FROM   ind_police_performance
""", conn), web, "police_performance")

save_table(q("""
    SELECT measure, year, crime_code, age, sex, cases, share, rate_per_10000,
           comparable_band
    FROM   ind_demographic_profile
""", conn), web, "demographic_profile")

save_table(q("""
    SELECT ine_code, year, population, total, violent, non_violent, unclassified,
           violent_ratio, violent_share, unclassified_share,
           shannon, shannon_normalised
    FROM   ind_crime_structure
""", conn), web, "crime_structure")

save_table(q("""
    SELECT ine_code, year, crime_code, offences, municipal_total,
           local_share, reference_share, location_quotient
    FROM   ind_crime_specialisation
""", conn), web, "crime_specialisation")


The `catalogue` file. The web needs it to build its menus
and to know which years each module covers, so no list of municipalities or
crime types is ever hard codedt.

In [ ]:
# Need it because the parent_code column is empty for some rows, which causes problems when converting to JSON
crime_types_reg = (
    q("""
        SELECT code, name_en, level, parent_code
        FROM crime_type_reg
        ORDER BY level, code
    """, conn)
    .astype(object)
    .where(lambda df: df.notna(), None)
    .to_dict(orient="records")
)

meta = {
    "municipalities": q("""
        SELECT ine_code, name, surface_km2 FROM municipality ORDER BY ine_code
    """, conn).to_dict(orient="records"),
    "crime_types_mun": q("""
        SELECT code, name_en, violence FROM crime_type_mun
        WHERE all_years ORDER BY code
    """, conn).to_dict(orient="records"),
    "crime_types_reg": crime_types_reg,
    "modules": {
        "A": {"name": "Crime level", "geography": "municipality",
              "years": year_range("ind_crime_level", conn)},
        "B": {"name": "Police performance",  "geography": "region",
              "years": year_range("ind_police_performance", conn)},
        "C": {"name": "Demographic profile", "geography": "region",
              "years": year_range("ind_demographic_profile", conn)},
        "D": {"name": "Crime structure and specialisation", "geography": "municipality",
              "years": year_range("ind_crime_structure", conn)}
    }
}

path = web / "meta.json"
path.write_text(json.dumps(meta, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
print(f"{path.name:34} {path.stat().st_size / 1024:>7.0f} KB")

## The municipal boundaries

Thing to take into account: 
- The map needs one polygon per municipality, in longitude and latitude, with the
INE code attached. 
- The IDEM shapefile coordinates are metres on ETRS89 UTM zone 30N
- The IDEM shapefile columns are called `CD_INE` and `DS_NOMBRE`

In [ ]:
gdf = gpd.read_file(SHAPEFILE, encoding="utf-8")
gdf = gdf.rename(columns={"CD_INE": "ine_code", "DS_NOMBRE": "name"})

gdf = gdf[gdf["DS_DESCRIP"].isin(("Término Municipal", "Termino Municipal"))].copy()

names = gdf.set_index("ine_code")["name"]

gdf["geometry"] = gdf.geometry.apply(lambda geometry: shapely.set_precision(geometry, 100))
gdf = gdf.dissolve(by="ine_code").reset_index()
gdf["name"] = gdf["ine_code"].map(names)
gdf = gdf[["ine_code", "name", "geometry"]]
gdf = gdf.to_crs(epsg=4326)

assert gdf["ine_code"].is_unique, "an INE code appears twice"
assert gdf["name"].notna().all(), "a municipality has no name"
assert not gdf.geometry.is_empty.any(), "a municipality has no geometry left"


gdf.to_file(OUTPUT_JSON, driver="GeoJSON", COORDINATE_PRECISION=5)
vertices = sum(len(g.exterior.coords) if g.geom_type == "Polygon" else sum(len(p.exterior.coords) for p in g.geoms) for g in gdf.geometry)

print(
  f"{len(gdf)} municipalities, about {vertices:,} outer vertices, "
  f"written to {OUTPUT_JSON}"
)

## The database as RDF

Data written into `rdf/madrid_crime.ttl`. 
Every row becomes a thing with an address, and the foreign keys of the database become links
between those things. Three parts for each column:

* **address**, built from its primary key
* **value**
* **link to another table**

In [ ]:
from rdflib import Graph, Literal, RDF, RDFS, URIRef
from src.utils import rows

# Change this to the address the project is published under. The whole point of
# RDF is that an identifier can be looked up.
BASE = "https://lauradiazm29.github.io/madrid-crime/id/"
PROP = BASE + "def#" 
MUNI = BASE + "municipality/"
TYPE_MUN = BASE + "offence-type/municipal/"
TYPE_REG = BASE + "offence-type/regional/"
OBS = BASE + "observation/"

g = Graph()
g.bind("def", PROP)
g.bind("mc", BASE)

print("ready")

### Master tables

In [ ]:
# municipality
for row in rows("SELECT ine_code, name, surface_km2 FROM municipality", conn):
  thing = URIRef(MUNI + row["ine_code"])
  g.add((thing, RDF.type, URIRef(PROP + "Municipality")))
  g.add((thing, RDFS.label, Literal(row["name"], lang="es")))
  g.add((thing, URIRef(PROP + "ineCode"), Literal(row["ine_code"])))
  g.add((thing, URIRef(PROP + "surfaceKm2"), Literal(row["surface_km2"])))

# crime_type_mun
for row in rows("SELECT code, name_en, all_years, violence FROM crime_type_mun", conn):
  thing = URIRef(TYPE_MUN + row["code"])
  g.add((thing, RDF.type, URIRef(PROP + "OffenceType")))
  g.add((thing, RDFS.label, Literal(row["name_en"], lang="en")))
  g.add((thing, URIRef(PROP + "comparableAcrossYears"), Literal(row["all_years"])))
  g.add((thing, URIRef(PROP + "violenceClass"), Literal(row["violence"])))

# crime_type_reg
for row in rows("SELECT code, name_en, level, parent_code, violence FROM crime_type_reg", conn):
  thing = URIRef(TYPE_REG + row["code"])
  g.add((thing, RDF.type, URIRef(PROP + "OffenceType")))
  g.add((thing, RDFS.label, Literal(row["name_en"], lang="en")))
  g.add((thing, URIRef(PROP + "level"), Literal(row["level"])))
  g.add((thing, URIRef(PROP + "violenceClass"), Literal(row["violence"])))
  if row["parent_code"] is not None:
      g.add((thing, URIRef(PROP + "partOf"), URIRef(TYPE_REG + row["parent_code"])))

print(f"{len(g):,} triples so far")

### Municipal Data

In [ ]:
# mun_population
for row in rows("SELECT ine_code, year, sex, municipality_population FROM mun_population", conn):
  thing = URIRef(f"{OBS}population/{row['ine_code']}/{row['year']}/{row['sex']}")
  g.add((thing, RDF.type, URIRef(PROP + "PopulationCount")))
  g.add((thing, URIRef(PROP + "municipality"), URIRef(MUNI + row["ine_code"])))
  g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
  g.add((thing, URIRef(PROP + "sex"), Literal(row["sex"])))
  g.add((thing, URIRef(PROP + "population"), Literal(row["municipality_population"])))

# mun_crime
for row in rows("SELECT ine_code, year, crime_code, crime_count FROM mun_crime", conn):
  thing = URIRef(f"{OBS}municipal-crime/{row['ine_code']}/{row['year']}/{row['crime_code']}")
  g.add((thing, RDF.type, URIRef(PROP + "MunicipalCrimeCount")))
  g.add((thing, URIRef(PROP + "municipality"), URIRef(MUNI + row["ine_code"])))
  g.add((thing, URIRef(PROP + "offenceType"),  URIRef(TYPE_MUN + row["crime_code"])))
  g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
  g.add((thing, URIRef(PROP + "offences"), Literal(row["crime_count"])))

print(f"{len(g):,} triples so far")

### Regional Data

In [ ]:
# reg_crime
for row in rows("SELECT year, crime_code, value FROM reg_crime", conn):
  thing = URIRef(f"{OBS}regional-crime/{row['year']}/{row['crime_code']}")
  g.add((thing, RDF.type, URIRef(PROP + "RegionalCrimeCount")))
  g.add((thing, URIRef(PROP + "offenceType"), URIRef(TYPE_MUN + row["crime_code"])))
  g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
  g.add((thing, URIRef(PROP + "offences"), Literal(row["value"])))

# reg_offences
for row in rows("SELECT year, crime_code, recorded, cleared FROM reg_offences", conn):
  thing = URIRef(f"{OBS}regional-offences/{row['year']}/{row['crime_code']}")
  g.add((thing, RDF.type, URIRef(PROP + "RecordedAndCleared")))
  g.add((thing, URIRef(PROP + "offenceType"), URIRef(TYPE_REG + row["crime_code"])))
  g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
  g.add((thing, URIRef(PROP + "recorded"), Literal(row["recorded"])))
  g.add((thing, URIRef(PROP + "cleared"), Literal(row["cleared"])))

# reg_victims
for row in rows("SELECT year, crime_code, age, sex, victims_count FROM reg_victims", conn):
  age = row["age"].replace("+", "-plus")
  thing = URIRef(f"{OBS}victims/{row['year']}/{row['crime_code']}/{age}/{row['sex']}")
  g.add((thing, RDF.type, URIRef(PROP + "VictimCount")))
  g.add((thing, URIRef(PROP + "offenceType"), URIRef(TYPE_REG + row["crime_code"])))
  g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
  g.add((thing, URIRef(PROP + "ageBand"), Literal(row["age"])))
  g.add((thing, URIRef(PROP + "sex"), Literal(row["sex"])))
  g.add((thing, URIRef(PROP + "people"), Literal(row["victims_count"])))

# reg_offenders
for row in rows("SELECT year, crime_code, age, sex, offenders_count FROM reg_offenders", conn):
  age = row["age"].replace("+", "-plus") #age band 65+ is written 65-plus
  thing = URIRef(f"{OBS}offenders/{row['year']}/{row['crime_code']}/{age}/{row['sex']}")
  g.add((thing, RDF.type, URIRef(PROP + "OffenderCount")))
  g.add((thing, URIRef(PROP + "offenceType"), URIRef(TYPE_REG + row["crime_code"])))
  g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
  g.add((thing, URIRef(PROP + "ageBand"), Literal(row["age"])))
  g.add((thing, URIRef(PROP + "sex"), Literal(row["sex"])))
  g.add((thing, URIRef(PROP + "people"), Literal(row["offenders_count"])))

print(f"{len(g):,} triples so far")

In [ ]:
# ind_crime_level
for row in rows("""SELECT ine_code, year, crime_code, denominator, offences, denom_value, rate, rate_per_1000, rate_per_100000 FROM ind_crime_level""", conn):
    thing = URIRef(f"{OBS}crime-level/{row['ine_code']}/{row['year']}/{row['crime_code']}/{row['denominator']}")
    g.add((thing, RDF.type, URIRef(PROP + "CrimeLevel")))
    g.add((thing, URIRef(PROP + "municipality"), URIRef(MUNI + row["ine_code"])))
    g.add((thing, URIRef(PROP + "offenceType"), URIRef(TYPE_MUN + row["crime_code"])))
    g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
    g.add((thing, URIRef(PROP + "denominator"), Literal(row["denominator"])))
    g.add((thing, URIRef(PROP + "denominatorValue"), Literal(row["denom_value"])))
    g.add((thing, URIRef(PROP + "offences"), Literal(row["offences"])))
    g.add((thing, URIRef(PROP + "rate"), Literal(row["rate"])))
    g.add((thing, URIRef(PROP + "ratePer1000"), Literal(row["rate_per_1000"])))
    g.add((thing, URIRef(PROP + "ratePer100000"), Literal(row["rate_per_100000"])))

# ind_police_performance
for row in rows("""SELECT year, crime_code, recorded, cleared, unsolved, clearance_rate, clearance_above_one, unsolved_share_in_level FROM ind_police_performance""", conn):
    thing = URIRef(f"{OBS}police-performance/{row['year']}/{row['crime_code']}")
    g.add((thing, RDF.type, URIRef(PROP + "PolicePerformance")))
    g.add((thing, URIRef(PROP + "offenceType"), URIRef(TYPE_REG + row["crime_code"])))
    g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
    g.add((thing, URIRef(PROP + "recorded"), Literal(row["recorded"])))
    g.add((thing, URIRef(PROP + "cleared"), Literal(row["cleared"])))
    g.add((thing, URIRef(PROP + "unsolved"), Literal(row["unsolved"])))
    g.add((thing, URIRef(PROP + "clearanceRate"), Literal(row["clearance_rate"])))
    g.add((thing, URIRef(PROP + "clearanceAboveOne"), Literal(row["clearance_above_one"])))
    g.add((thing, URIRef(PROP + "unsolvedShareInLevel"), Literal(row["unsolved_share_in_level"])))

# Indicator C, victims and offenders by age and sex
for row in rows("""SELECT measure, year, crime_code, age, sex, cases, share, rate_per_10000, is_age_total, comparable_band FROM ind_demographic_profile""", conn):
    age = row["age"].replace("+", "-plus")
    thing = URIRef(f"{OBS}demographic-profile/{row['measure']}/{row['year']}/{row['crime_code']}/{age}/{row['sex']}")
    g.add((thing, RDF.type, URIRef(PROP + "DemographicProfile")))
    g.add((thing, URIRef(PROP + "offenceType"), URIRef(TYPE_REG + row["crime_code"])))
    g.add((thing, URIRef(PROP + "peopleCounted"), Literal(row["measure"])))
    g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
    g.add((thing, URIRef(PROP + "ageBand"), Literal(row["age"])))
    g.add((thing, URIRef(PROP + "sex"), Literal(row["sex"])))
    g.add((thing, URIRef(PROP + "people"), Literal(row["cases"])))
    g.add((thing, URIRef(PROP + "share"), Literal(row["share"])))
    g.add((thing, URIRef(PROP + "ratePer10000"), Literal(row["rate_per_10000"])))
    g.add((thing, URIRef(PROP + "isAgeTotal"), Literal(row["is_age_total"])))
    g.add((thing, URIRef(PROP + "comparableAgeBand"), Literal(row["comparable_band"])))

# ind_crime_structure
for row in rows("""SELECT ine_code, year, population, total, violent, non_violent, unclassified, crime_types_count, violent_ratio,
                          violent_share, unclassified_share, shannon, shannon_normalised FROM ind_crime_structure""", conn):
    thing = URIRef(f"{OBS}crime-structure/{row['ine_code']}/{row['year']}")
    g.add((thing, RDF.type, URIRef(PROP + "CrimeStructure")))
    g.add((thing, URIRef(PROP + "municipality"), URIRef(MUNI + row["ine_code"])))      
    g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
    g.add((thing, URIRef(PROP + "population"), Literal(row["population"])))
    g.add((thing, URIRef(PROP + "offences"), Literal(row["total"])))
    g.add((thing, URIRef(PROP + "violentOffences"), Literal(row["violent"])))
    g.add((thing, URIRef(PROP + "nonViolentOffences"), Literal(row["non_violent"])))
    g.add((thing, URIRef(PROP + "unclassifiedOffences"), Literal(row["unclassified"])))
    g.add((thing, URIRef(PROP + "offenceTypesPresent"), Literal(row["crime_types_count"])))
    g.add((thing, URIRef(PROP + "violentRatio"), Literal(row["violent_ratio"])))
    g.add((thing, URIRef(PROP + "violentShare"), Literal(row["violent_share"])))
    g.add((thing, URIRef(PROP + "unclassifiedShare"), Literal(row["unclassified_share"])))
    g.add((thing, URIRef(PROP + "shannonEntropy"), Literal(row["shannon"])))
    g.add((thing, URIRef(PROP + "shannonNormalised"), Literal(row["shannon_normalised"])))

# ind_crime_specialisation
for row in rows("""SELECT ine_code, year, crime_code, offences, municipal_total, local_share, reference_share, location_quotient FROM ind_crime_specialisation""", conn):
    thing = URIRef(f"{OBS}specialisation/{row['ine_code']}/{row['year']}/{row['crime_code']}")
    g.add((thing, RDF.type, URIRef(PROP + "CrimeSpecialisation")))
    g.add((thing, URIRef(PROP + "municipality"), URIRef(MUNI + row["ine_code"])))
    g.add((thing, URIRef(PROP + "offenceType"), URIRef(TYPE_MUN + row["crime_code"])))
    g.add((thing, URIRef(PROP + "year"), Literal(row["year"])))
    g.add((thing, URIRef(PROP + "offences"), Literal(row["offences"])))
    g.add((thing, URIRef(PROP + "totalOffences"), Literal(row["municipal_total"])))
    g.add((thing, URIRef(PROP + "localShare"), Literal(row["local_share"])))
    g.add((thing, URIRef(PROP + "referenceShare"), Literal(row["reference_share"])))
    g.add((thing, URIRef(PROP + "locationQuotient"), Literal(row["location_quotient"])))

print(f"{len(g):,} triples in total")

In [ ]:
rdf_dir = Path(RDF_DIR)
rdf_dir.mkdir(parents=True, exist_ok=True)

turtle_path = rdf_dir / "madrid_crime.ttl"
g.serialize(destination=turtle_path, format="turtle")
print(f"{turtle_path.name:24} {turtle_path.stat().st_size / 1024 / 1024:>6.1f} MB")

In [ ]:
conn.close()
print("done")